In [2]:
import pandas as pd

In [4]:
customers = pd.read_json('D:\\CODING\\data mining\\lab 3\\customers_dirty.json')
transactions = pd.read_csv('D:\\CODING\\data mining\\lab 3\\transactions_dirty.csv')
print("loaded")


loaded


Missing ages
Categorical inconsistencies (we saw earlier: Mle, m, FEMALE etc.)
Possible unrealistic ages (we will check next)

# Duplicate Detection in Transactions

In [7]:
# Check number of duplicate rows
duplicate_count = transactions.duplicated().sum()
print("Number of duplicate transaction records:", duplicate_count)

# Display duplicate rows (if any)
duplicates = transactions[transactions.duplicated()]
print("\nDuplicate Records:\n", duplicates)


Number of duplicate transaction records: 20

Duplicate Records:
     CustomerID  Monthly_Spend  Account_Balance  Trans_Count
250       C070         916.02         14505.09           21
253       C205        3929.37          3909.08           11
254       C247        3565.07         18821.21           79
255       C178        4116.77         19595.08           31
257       C174        4471.06         15173.47           82
258       C075        4448.97         15329.56           19
260       C164        3551.26         18354.96            4
261       C065        2863.90         14421.56           93
262       C209        3393.06          3036.49           35
263       C233        4574.45          7323.46           74
264       C168        2853.73         15350.83           77
265       C204        1550.51          3077.42           25
266       C146        1767.71         17335.30           96
267       C106        2342.97          4350.45           99
268       C105        3191.12      

In [8]:
# Remove duplicates
transactions_clean = transactions.drop_duplicates()

print("Shape before removing duplicates:", transactions.shape)
print("Shape after removing duplicates:", transactions_clean.shape)


Shape before removing duplicates: (275, 4)
Shape after removing duplicates: (255, 4)


# Handle Missing Financial Values

In [10]:
#check missing values again

print("\nMissing Values Before Cleaning:\n")
print(transactions_clean.isnull().sum())



Missing Values Before Cleaning:

CustomerID          0
Monthly_Spend      15
Account_Balance    10
Trans_Count         0
dtype: int64


In [14]:
# Remove duplicates and create a proper copy
transactions_clean = transactions.drop_duplicates().copy()


In [16]:
# Check missing values
print(transactions_clean.isnull().sum())

# Fill missing Monthly_Spend
monthly_median = transactions_clean["Monthly_Spend"].median()
transactions_clean.loc[:, "Monthly_Spend"] = transactions_clean["Monthly_Spend"].fillna(monthly_median)

# Fill missing Account_Balance
balance_median = transactions_clean["Account_Balance"].median()
transactions_clean.loc[:, "Account_Balance"] = transactions_clean["Account_Balance"].fillna(balance_median)

# Check again
print(transactions_clean.isnull().sum())


CustomerID          0
Monthly_Spend       0
Account_Balance    10
Trans_Count         0
dtype: int64
CustomerID         0
Monthly_Spend      0
Account_Balance    0
Trans_Count        0
dtype: int64


In [17]:
#Replace invalid ages
# Create a copy to avoid warning
customers_clean = customers.copy()

# Replace negative ages and unrealistic ages (>100) with NaN
customers_clean.loc[
    (customers_clean["Age"] < 0) | (customers_clean["Age"] > 100),
    "Age"
] = None

# Check how many missing values now
print("Missing values after replacing invalid ages:")
print(customers_clean.isnull().sum())


Missing values after replacing invalid ages:
CustomerID       0
Age             15
Gender           0
Credit_Score     0
Region           0
dtype: int64


In [18]:
# Calculate median age
age_median = customers_clean["Age"].median()

# Fill missing Age with median
customers_clean["Age"] = customers_clean["Age"].fillna(age_median)

# Check again
print("Missing values after filling Age:")
print(customers_clean.isnull().sum())


Missing values after filling Age:
CustomerID      0
Age             0
Gender          0
Credit_Score    0
Region          0
dtype: int64


In [19]:
#Clean Gender colum
# Convert all gender values to lowercase
customers_clean["Gender"] = customers_clean["Gender"].str.lower()

# Replace inconsistent values
customers_clean["Gender"] = customers_clean["Gender"].replace({
    "m": "male",
    "mle": "male",
    "f": "female"
})

# Check unique values
print("Unique gender values after cleaning:")
print(customers_clean["Gender"].unique())


Unique gender values after cleaning:
['male' 'female']


In [20]:
#Now we clean the Region column
# Convert region values to lowercase first
customers_clean["Region"] = customers_clean["Region"].str.lower()

# Capitalize first letter
customers_clean["Region"] = customers_clean["Region"].str.capitalize()

# Check unique values
print("Unique region values after cleaning:")
print(customers_clean["Region"].unique())




Unique region values after cleaning:
['South' 'East' 'West' 'North']


# Scaling

In [26]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

transactions_clean[["Monthly_Spend", "Account_Balance"]] = scaler.fit_transform(
    transactions_clean[["Monthly_Spend", "Account_Balance"]]
)


In [28]:
print(transactions_clean[["Monthly_Spend", "Account_Balance"]].describe())


       Monthly_Spend  Account_Balance
count     255.000000       255.000000
mean        0.511707         0.493746
std         0.292223         0.291578
min         0.000000         0.000000
25%         0.252656         0.255100
50%         0.533347         0.496115
75%         0.756044         0.734186
max         1.000000         1.000000


In [29]:
print("Monthly_Spend Min:", transactions_clean["Monthly_Spend"].min())
print("Monthly_Spend Max:", transactions_clean["Monthly_Spend"].max())

print("Account_Balance Min:", transactions_clean["Account_Balance"].min())
print("Account_Balance Max:", transactions_clean["Account_Balance"].max())


Monthly_Spend Min: 0.0
Monthly_Spend Max: 1.0
Account_Balance Min: 0.0
Account_Balance Max: 1.0


# Encoding Gender

In [27]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

customers_clean["Gender"] = encoder.fit_transform(customers_clean["Gender"])


In [30]:
print(customers_clean["Gender"].unique())


[1 0]


In [31]:
print(final_dataset.head())


  CustomerID  Monthly_Spend  Account_Balance  Trans_Count   Age  Gender  \
0       C122         318.03          5289.30           79  77.0  female   
1       C138         513.40         11277.11           13  87.0    male   
2       C155        2649.25         13616.40           79  58.0    male   
3       C058         595.71          9218.01           49  50.0    male   
4       C046          62.14         16750.90          100  23.0  female   

   Credit_Score Region  
0           521   East  
1           622   East  
2           474  North  
3           707   East  
4           713  North  


In [32]:
# Merge transactions and customers data using CustomerID

final_dataset = pd.merge(
    transactions_clean,      # first dataframe
    customers_clean,         # second dataframe
    on="CustomerID",         # common column
    how="inner"              # keep only matching records
)

# Check first 5 rows
print(final_dataset.head())

# Check shape of merged dataset
print("Shape of merged dataset:", final_dataset.shape)


  CustomerID  Monthly_Spend  Account_Balance  Trans_Count   Age  Gender  \
0       C122       0.061840         0.266180           79  77.0       0   
1       C138       0.101086         0.568602           13  87.0       1   
2       C155       0.530131         0.686751           79  58.0       1   
3       C058       0.117620         0.464605           49  50.0       1   
4       C046       0.010438         0.845063          100  23.0       0   

   Credit_Score Region  
0           521   East  
1           622   East  
2           474  North  
3           707   East  
4           713  North  
Shape of merged dataset: (255, 8)


In [34]:
final_dataset.to_csv("final_dataset.csv", index=False)
